# Paneles solares: YOLO por fuente

Tres arquitecturas (`yolov8n.pt`, `yolo11n.pt`, `yolo26n.pt`), tres estrategias de congelamiento y tres folds. Son 27 entrenamientos. La métrica principal es el promedio macro de mAP50-95, con el mismo peso para cada fold.

Elegí una GPU NVIDIA en *Entorno de ejecución*. Los pesos `best.pt` se copian a `RESULTS_DIR`, que tiene que vivir en Google Drive.

Las celdas de rutas, instalación y verificación no cargan modelos. La última celda sí entrena.

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

In [ ]:
from pathlib import Path

# Editar estas tres rutas. No hay una carpeta de Drive fija en el código.
if IN_COLAB:
    REPO_DIR = Path("/content/drive/MyDrive/procesamiento-de-imagenes")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "YOLO" else Path.cwd()

CODE_DIR = REPO_DIR / "YOLO"
DATASET_DIR = REPO_DIR / "data" / "yolo"
RESULTS_DIR = REPO_DIR / "YOLO" / "results"

assert CODE_DIR.joinpath("train.py").is_file(), f"No está train.py en {CODE_DIR}"
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("código   :", CODE_DIR)
print("dataset  :", DATASET_DIR)
print("resultados:", RESULTS_DIR)

In [ ]:
import sys

# No reinstala torch: Colab ya trae la build con CUDA.
!{sys.executable} -m pip install -q -r "{CODE_DIR / 'requirements.txt'}"

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Activá una GPU en el entorno de ejecución antes de entrenar. "
        "No se cargó ningún modelo."
    )
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
import sys
sys.path.insert(0, str(CODE_DIR))

from solar_yolo.config import Config
from solar_yolo.engine import verify_setup

config = Config(dataset_dir=DATASET_DIR, results_dir=RESULTS_DIR, device=0, batch=16)
print(verify_setup(config))

## Entrenar

`None` recorre las 27 combinaciones. Para una sola, pasá una lista.

Si Colab se queda sin memoria, bajá `batch` en la celda anterior y volvé a correr desde ahí. Un experimento ya terminado, con su `best.pt`, no se repite.

In [ ]:
from solar_yolo.engine import run_experiments

SELECTED_ARCHITECTURES = None  # ["yolov8n", "yolo11n", "yolo26n"]
SELECTED_STRATEGIES = None     # ["full", "backbone", "partial"]
SELECTED_FOLDS = None          # [1, 2, 3]

outcome = run_experiments(
    config,
    architectures=SELECTED_ARCHITECTURES,
    strategies=SELECTED_STRATEGIES,
    folds=SELECTED_FOLDS,
)
print("fallos:", len(outcome["failures"]))
for failure in outcome["failures"]:
    print(failure["experiment_id"], failure["error"])